In [1]:
#include "DHT.h"
#include <ESP8266WiFi.h>
#include <WiFiClient.h>
#include <LiquidCrystal_I2C.h>

#define DHTPIN D3
#define airquality_pin A0
#define ir_pin D0
#define buzzer D5
#define DHTTYPE DHT11

DHT dht(DHTPIN, DHTTYPE);
LiquidCrystal_I2C lcd(0x27, 16, 2); // D1-SCL D2-SDA

float humidityData;
float temperatureData;
int airqualityData;
int irData;
int time1;

const char* ssid = "YOUR_WIFI_NAME";
const char* password = "YOUR_WIFI_PASSWORD";
const char* apiKey = "YOUR_THINGSPEAK_API_KEY";

char server[] = "api.thingspeak.com";

byte degree_symbol[8] = {
  0b00111,
  0b00101,
  0b00111,
  0b00000,
  0b00000,
  0b00000,
  0b00000,
  0b00000
};

WiFiClient client;

void setup() {
  Serial.begin(115200);

  lcd.init();
  lcd.backlight();
  lcd.setCursor(4, 0);
  lcd.print("Coal Mine");
  lcd.setCursor(3, 1);
  lcd.print("Monitoring");
  delay(2000);
  lcd.clear();

  dht.begin();

  Serial.print("Connecting to ");
  Serial.println(ssid);

  WiFi.begin(ssid, password);

  while (WiFi.status() != WL_CONNECTED) {
    delay(500);
    Serial.print(".");
    lcd.setCursor(0, 0);
    lcd.print("Connecting WiFi");
  }

  lcd.clear();
  lcd.print("WiFi Connected");
  delay(1000);

  pinMode(airquality_pin, INPUT);
  pinMode(ir_pin, INPUT);
  pinMode(buzzer, OUTPUT);
  digitalWrite(buzzer, LOW);
}

void loop() {

  humidityData = dht.readHumidity();
  temperatureData = dht.readTemperature();
  airqualityData = analogRead(airquality_pin);
  irData = digitalRead(ir_pin);

  if (irData == 1) {
    digitalWrite(buzzer, LOW);
    time1 = 10000;
    sendToThingSpeak();
  } 
  else {
    time1 = 1000;

    lcd.clear();
    lcd.setCursor(0, 0);
    lcd.print("Collision");
    lcd.setCursor(0, 1);
    lcd.print("Detected!");

    for (int i = 0; i < 6; i++) {
      digitalWrite(buzzer, HIGH);
      delay(100);
      digitalWrite(buzzer, LOW);
      delay(100);
    }
  }

  delay(time1);
}

void sendToThingSpeak() {

  if (client.connect(server, 80)) {

    String url = "/update?api_key=" + String(apiKey);
    url += "&field1=" + String(humidityData);
    url += "&field2=" + String(temperatureData);
    url += "&field3=" + String(airqualityData);
    url += "&field4=" + String(irData);

    client.print(String("GET ") + url + " HTTP/1.1\r\n" +
                 "Host: api.thingspeak.com\r\n" +
                 "Connection: close\r\n\r\n");

    lcd.clear();
    lcd.setCursor(0, 0);
    lcd.print("Humidity:");
    lcd.print(humidityData);
    lcd.print("%");

    delay(2000);

    lcd.clear();
    lcd.setCursor(0, 0);
    lcd.print("Temp:");
    lcd.print(temperatureData);
    lcd.createChar(1, degree_symbol);
    lcd.setCursor(10, 0);
    lcd.write(1);
    lcd.print("C");

    delay(2000);

    lcd.clear();
    lcd.setCursor(0, 0);
    lcd.print("Air Quality:");
    lcd.setCursor(0, 1);
    lcd.print(airqualityData);
    lcd.print(" PPM");

    delay(2000);

    lcd.clear();
    lcd.setCursor(0, 0);
    lcd.print("Data Sending...");

  } 
  else {
    Serial.println("Connection Failed");
  }

  client.stop();
}

SyntaxError: invalid syntax (3494001036.py, line 12)